In [2]:
import pygame
import sys
import math
import heapq

#Button Class
class Button:
    def __init__(self, x, y, width, height, text):
        self.rect=pygame.Rect(x, y, width, height)
        self.text=text
        self.selected=False

    def draw(self, screen):
        color=(180, 180, 180) if self.selected else (220, 220, 220)
        pygame.draw.rect(screen, color, self.rect)
        pygame.draw.rect(screen, BLACK, self.rect, 2)

        font =pygame.font.SysFont(None, 24)
        textSurface=font.render(self.text, True, BLACK)
        screen.blit(textSurface, (self.rect.x+10, self.rect.y+10))

    def isClicked(self, pos):
        return self.rect.collidepoint(pos)


#Row and Col input

while True:
    rows=input('Enter number of rows(10-60):')
    try:
        rows=int(rows)
        if rows<10 or rows>60:
            print('Number out of range!')
        else:
            break
    except ValueError:
        print('Invalid Input!')

while True:
    cols=input('Enter number of columns(10-60):')
    try:
        cols=int(cols)
        if cols<10 or cols>60:
            print('Number out of range!')
        else:
            break
    except ValueError:
        print('Invalid Input!')

pygame.init()


#Buttons initialization

manhattanBtn = Button(620, 50, 160, 40, "Manhattan")
euclideanBtn = Button(620, 100, 160, 40, "Euclidean")

astarBtn = Button(620, 200, 160, 40, "A*")
GBFSBtn = Button(620, 250, 160, 40, "GBFS")

staticBtn = Button(620, 350, 160, 40, "Static")
dynamicBtn = Button(620, 400, 160, 40, "Dynamic")

startBtn = Button(620, 500, 160, 40, 'Start')
pauseBtn = Button(620, 550, 160, 40, 'Pause')
stopBtn = Button(620, 600, 160, 40, 'Stop')

#Constants

EMPTY=0
WALL=1
START=2
GOAL=3
PATH=4
EXPLORED=5

WHITE=(255, 255, 255)
BLACK=(0,0,0)
GRAY=(100, 100, 100)
GREEN=(0, 255, 0)
RED=(255, 0, 0)
BLUE=(0, 0, 255)
YELLOW=(255, 255, 0)

GRID_WIDTH=600
PANEL_WIDTH=200
WINDOW_HEIGHT=800

#Default Settings

cellSize=GRID_WIDTH//max(rows, cols)
screen = pygame.display.set_mode((GRID_WIDTH + PANEL_WIDTH, WINDOW_HEIGHT))
pygame.display.set_caption('Dynamic Pathfinding Grid')

grid = [[0 for _ in range(cols)] for _ in range(rows)]

startPosition=None
goalPosition=None

environmentType='static'
mode='wall'
heuristicType='manhattan'

algorithmRunning=False
algorithmPaused=False
frontier=[]
parent={}
gValue={}

#A* Search

def manhattan(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def euclidean(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

def initialze_aStar():
    global frontier, parent, gValue
    frontier=[]
    parent={}
    gValue={startPosition:0}
    heapq.heappush(frontier, (0, startPosition))

def a_star_step():
    global frontier

    if not frontier:
        return False
    
    
    current=heapq.heappop(frontier)[1]

    r, c=current
    if grid[r][c] not in (START, GOAL):
        grid[r][c]=EXPLORED

    if current==goalPosition:
        reconstructPath(parent, current)
        return False

    neighbours=get_neighbours(current)

    for neighbour in neighbours:
        tempGValue=gValue[current]+1
        if neighbour not in gValue or tempGValue<gValue[neighbour]:
            parent[neighbour]=current
            gValue[neighbour]=tempGValue

            if heuristicType=='manhattan':
                hValue=manhattan(neighbour, goalPosition)
            else:
                hValue=euclidean(neighbour, goalPosition)
            fValue=tempGValue+hValue
            heapq.heappush(frontier, (fValue, neighbour))

            r, c=neighbour
            if grid[r][c]==EMPTY:
                grid[r][c]=EXPLORED
    return True

def get_neighbours(node):
    neighbours=[]
    row, col=node
    directions=[(1, 0), (-1, 0), (0, 1), (0, -1)]

    for directionRow, directionCol in directions:
        nextRow, nextCol=row+directionRow, col+directionCol

        if 0<=nextRow<rows and 0<=nextCol<cols:
            if grid[nextRow][nextCol]!=WALL:
                neighbours.append((nextRow, nextCol))
    return neighbours

def reconstructPath(parent, current):
    while current in parent:
        current=parent[current]
        if current!=startPosition:
            row, col = current
            grid[row][col]=PATH




#Main Loop



running =True
while running:
    screen.fill(WHITE)

    pygame.draw.rect(screen, (240,240,240), (600, 0, 200, 800))

    manhattanBtn.draw(screen)
    euclideanBtn.draw(screen)
    astarBtn.draw(screen)
    GBFSBtn.draw(screen)
    staticBtn.draw(screen)
    dynamicBtn.draw(screen)
    startBtn.draw(screen)
    pauseBtn.draw(screen)
    stopBtn.draw(screen)

    #Assigning colosr to different types of cells
    
    for i in range(rows):
        for j in range(cols):
            color=WHITE
            if grid[i][j]==WALL:
                color=GRAY
            elif grid[i][j]==START:
                color=GREEN
            elif grid[i][j]==GOAL:
                color=RED
            elif grid[i][j]==PATH:
                color=BLUE
            elif grid[i][j]==EXPLORED:
                color=YELLOW
            rectangle=pygame.Rect(j*cellSize, i*cellSize, cellSize, cellSize)
            pygame.draw.rect(screen, color, rectangle)
            pygame.draw.rect(screen, BLACK, rectangle, 1)


    #Mouse Clicking and Key Pressing handling
    
    for event in pygame.event.get():
        if event.type==pygame.QUIT:
            running=False
        if event.type==pygame.MOUSEBUTTONDOWN:
            pos=pygame.mouse.get_pos()


            if manhattanBtn.isClicked(pos):
                heuristicType='manhattan'
                manhattanBtn.selected=True
                euclideanBtn.selected=False

            elif euclideanBtn.isClicked(pos):
                heuristicType='euclidean'
                euclideanBtn.selected=True
                manhattanBtn.selected=False
            elif astarBtn.isClicked(pos):
                algorithmType = "astar"
                astarBtn.selected = True
                GBFSBtn.selected = False

            elif GBFSBtn.isClicked(pos):
                algorithmType = "GBFS"
                GBFSBtn.selected = True
                astarBtn.selected = False

            elif staticBtn.isClicked(pos):
                environmentType = "static"
                staticBtn.selected = True
                dynamicBtn.selected = False

            elif dynamicBtn.isClicked(pos):
                environmentType = "dynamic"
                dynamicBtn.selected = True
                staticBtn.selected = False
            elif startBtn.isClicked(pos):
                if startPosition and goalPosition:
                    initialze_aStar()
                    algorithmRunning=True
                    algorithmPaused=False
            elif pauseBtn.isClicked(pos):
                algorithmPaused=not algorithmPaused
            elif stopBtn.isClicked(pos):
                algorithmRunning=False
                algorithmPaused=False

            
            x, y=pygame.mouse.get_pos()

            if x<GRID_WIDTH and y<rows*cellSize:
                row=y//cellSize
                col=x//cellSize

                if mode=='wall':
                    if grid[row][col]==EMPTY:
                        grid[row][col]=WALL
                    elif grid[row][col]==WALL:
                        grid[row][col]=EMPTY

                elif mode=='start':
                    if startPosition:
                        prevRow, prevCol=startPosition
                        grid[prevRow][prevCol]=EMPTY
                    startPosition=(row, col)
                    grid[row][col]=START
                elif mode=='goal':
                    if goalPosition:
                        prevRow, prevCol=goalPosition
                        grid[prevRow][prevCol]=EMPTY
                    goalPosition=(row, col)
                    grid[row][col]=GOAL
                
        if event.type==pygame.KEYDOWN:
            if event.key==pygame.K_s:
                mode='start'
            elif event.key==pygame.K_g:
                mode='goal'
            elif event.key==pygame.K_w:
                mode='wall'
        
    pygame.display.flip()

    if algorithmRunning and not algorithmPaused:
        stillRunning=a_star_step()
        if not stillRunning:
            algorithmRunning=False

pygame.quit()
    

Enter number of rows(10-60): 20
Enter number of columns(10-60): 20
